# Python `defaultdict`

1. 為什麼需要 `defaultdict`
2. `defaultdict(int)`、`defaultdict(list)`、`defaultdict(lambda: ...)` 的用法
3. 字典排序技巧與字典推導式
4. 用 `defaultdict` 構建圖的鄰接表、分組數據
5. 用 `defaultdict` 解 AtCoder ABC170 D 類型題目


## 1. 為什麼需要 `defaultdict`？

一般的 `dict`，如果讀取「不存在的 key」，會直接出現 `KeyError`。

所以計數時常常要先檢查 key 存不存在，程式會變長。


In [ ]:
d = {}

try:
    d["a"] += 1
except KeyError as e:
    print("發生 KeyError:", e)

# 傳統寫法：要先檢查 key 是否存在
d = {}
s = "hello"
for ch in s:
    if ch in d:
        d[ch] += 1
    else:
        d[ch] = 1
print(d)


## 2. `defaultdict` 基本用法

`defaultdict` 在建立時要給一個「工廠函式」，
當讀取不存在的 key 時，會自動用這個函式產生預設值。

```python
from collections import defaultdict
dd = defaultdict(int)                    # 預設值為 0
dd = defaultdict(list)                   # 預設值為空列表
dd = defaultdict(lambda: float('inf'))   # 預設值為無限大
```


In [ ]:
from collections import defaultdict

# 預設值為 0：計數不用再檢查 key
dd = defaultdict(int)
s = "hello"
for ch in s:
    dd[ch] += 1
print(dd)


In [ ]:
# 預設值為空列表：直接 append
dd = defaultdict(list)
dd["fruits"].append("apple")
dd["fruits"].append("banana")
print(dd)

# 預設值為無限大：常用在最短路徑 (Dijkstra)
dist = defaultdict(lambda: float('inf'))
dist["start"] = 0
print(dist["start"])
print(dist["尚未拜訪的節點"])


## 3. 自訂工廠函式

工廠函式也可以自己定義，回傳任何你要的預設值。


In [ ]:
from collections import defaultdict

def fa():
    return 100

d = defaultdict(fa)
d[10] = 5
print(d[10])   # 已經指定過 → 5
print(d[2])    # 沒指定過 → 自動用 fa() 產生 100


### 注意：「讀取」不存在的 key 也會自動建立該 key

這是 `defaultdict` 常見的陷阱：只是查看一下，key 就被建立了。


In [ ]:
from collections import defaultdict

d = defaultdict(int)
print("一開始的長度:", len(d))

_ = d["x"]   # 只是讀取，沒有指定
print("讀取 d['x'] 之後的長度:", len(d))
print(d)

# 如果只想「查看」而不建立 key，用 in 或 .get()
print("y" in d)
print(d.get("y", 0))
print("用 get 之後的長度:", len(d))


## 4. 範例：計算字母出現次數


In [ ]:
from collections import defaultdict

letter_count = defaultdict(int)
sentence = "hello world"

for letter in sentence:
    if letter != " ":
        letter_count[letter] += 1

for k, v in letter_count.items():
    print(k, v)


In [ ]:
# 工廠函式用 lambda 的寫法
from collections import defaultdict

s = "hello world"
b = defaultdict(lambda: 0)
for i in s:
    b[i] += 1
print(b)

# 預設值也可以是字串
c = defaultdict(lambda: "no")
print(c["沒出現過的字"])


## 5. 字典排序技巧

`sorted()` 搭配 `key` 參數，可以用不同規則排序字典。


In [ ]:
d = {"banana": 3, "apple": 5, "cherry": 3}

# 依 key 排序
print(sorted(d.items()))

# 依 value 排序
print(sorted(d.items(), key=lambda x: x[1]))

# 依 value 由大到小，value 相同時依 key 由小到大
print(sorted(d.items(), key=lambda x: (-x[1], x[0])))


## 6. 字典推導式與條件過濾


In [ ]:
# 字典推導式
d = {k: v**2 for k, v in enumerate(range(5))}
print(d)

# 條件過濾：只留下 value 是偶數的
d2 = {k: v for k, v in d.items() if v % 2 == 0}
print(d2)


## 7. 構建圖的鄰接表

`defaultdict(list)` 是圖論題最常用的工具：
不用先建立每個節點，直接 append 邊即可。


In [ ]:
from collections import defaultdict

graph = defaultdict(list)
edges = [(1, 2), (2, 3), (1, 3)]

for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)   # 如果是無向圖

for node in sorted(graph):
    print(node, "->", graph[node])


## 8. 分組數據


In [ ]:
from collections import defaultdict

data = [("apple", 2), ("banana", 3), ("apple", 5)]
group = defaultdict(list)

for key, value in data:
    group[key].append(value)

print(group)


## 9. AtCoder ABC170 D - Not Divisible

題目連結：<https://atcoder.jp/contests/abc170/tasks/abc170_d>

【題目描述】

給定一個包含 $N$ 個整數的數列 $A$，找出數列中有多少個數 $A_i$ 滿足：
對於所有 $j \neq i$，$A_j$ 都不能整除 $A_i$。

【輸入】

第一列：一個整數 $N$。
第二列：$N$ 個整數 $A_1, A_2, \dots, A_N$（$1 \le A_i \le 10^6$）。

【輸出】

滿足條件的數的個數。

【範例輸入】

```text
5
24 11 8 3 16
```

【範例輸出】

```text
3
```

說明：24 被 8 和 3 整除（不算）、16 被 8 整除（不算），
剩下 11、8、3 沒有被其他元素整除 → 答案是 3。


In [ ]:
# 用字串模擬輸入
sample = """5
24 11 8 3 16"""

data = sample.split("\n")
print(data)


## 10. 解題想法

暴力法要兩兩比較，$O(N^2)$ 太慢（$N$ 可達 $2 \times 10^5$）。

改用「篩法」的想法：

1. 用 `defaultdict(int)` 統計每個數字出現幾次
2. 對每個出現過的數字 $v$，把 $2v, 3v, 4v, \dots$ 全部標記為「會被整除」
3. 一個數 $A_i$ 合格的條件：
   - 沒有被任何其他數的倍數標記到
   - 而且自己只出現一次（出現兩次以上會被「另一個自己」整除）


In [ ]:
from collections import defaultdict

def solve(data):
    n = int(data[0])
    a = list(map(int, data[1].split()))

    cnt = defaultdict(int)
    for x in a:
        cnt[x] += 1

    max_a = max(a)
    divisible = [False] * (max_a + 1)   # divisible[x] = x 會被某個元素整除

    for v in cnt:                        # 只對「出現過的數字」做篩法
        for multiple in range(2 * v, max_a + 1, v):
            divisible[multiple] = True

    ans = 0
    for x in a:
        if not divisible[x] and cnt[x] == 1:
            ans += 1
    return ans

print(solve(data))   # 應該輸出 3


## 11. 比賽提交版

```python
import sys
from collections import defaultdict

def main():
    data = sys.stdin.read().split()
    n = int(data[0])
    a = list(map(int, data[1:1 + n]))

    cnt = defaultdict(int)
    for x in a:
        cnt[x] += 1

    max_a = max(a)
    divisible = [False] * (max_a + 1)
    for v in cnt:
        for multiple in range(2 * v, max_a + 1, v):
            divisible[multiple] = True

    ans = 0
    for x in a:
        if not divisible[x] and cnt[x] == 1:
            ans += 1
    print(ans)

main()
```


## 12. 練習題

### 題目 1：單字分組

給定一串單字，把「第一個字母相同」的單字分在同一組，
依字母順序輸出每一組。

輸入：`apple banana cherry avocado blueberry`

期望輸出：

```text
a ['apple', 'avocado']
b ['banana', 'blueberry']
c ['cherry']
```

### 題目 2：成績統計

每列資料是「姓名 分數」，同一個人可能出現多次。
請輸出每個人的平均分數（依姓名排序）。

輸入：

```text
Amy 80
Bob 70
Amy 90
Bob 90
```

期望輸出：

```text
Amy 85.0
Bob 80.0
```


In [ ]:
# 練習作答區：題目 1
words = "apple banana cherry avocado blueberry".split()

# 提示：defaultdict(list)，key 用 word[0]


In [ ]:
# 練習作答區：題目 2
scores = ["Amy 80", "Bob 70", "Amy 90", "Bob 90"]

# 提示：defaultdict(list)，最後用 sum(v)/len(v) 算平均


### 參考答案


In [ ]:
from collections import defaultdict

# 題目 1
words = "apple banana cherry avocado blueberry".split()
group = defaultdict(list)
for w in words:
    group[w[0]].append(w)
for k in sorted(group):
    print(k, group[k])


In [ ]:
from collections import defaultdict

# 題目 2
scores = ["Amy 80", "Bob 70", "Amy 90", "Bob 90"]
record = defaultdict(list)
for line in scores:
    name, s = line.split()
    record[name].append(int(s))
for name in sorted(record):
    v = record[name]
    print(name, sum(v) / len(v))
